In [1]:
import re

In [ ]:
data = {"idx": "5855", "commit_id": "d9bce9d99f4656ae0b0127f7472db9067b8f84ab", "project": "qemu", "func_name": "PPC_OP", "language": "cpp", "code": "PPC_OP(slw)\n\n{\n\n    if (T1 & 0x20) {\n\n        T0 = 0;\n\n    } else {\n\n        T0 = T0 << T1;\n\n    }\n\n    RETURN();\n\n}\n", "code_tokens": ["PPC_OP", "(", "slw", ")", "{", "if", "(", "T1", "&", "0x20", ")", "{", "T0", "=", "0", ";", "}", "else", "{", "T0", "=", "T0", "<<", "T1", ";", "}", "RETURN", "(", ")", ";", "}"], "variables": ["PPC_OP", "T1", "T0", "T1"], "label": 1}
code = data['code']

In [ ]:
print(code)

static void v9fs_getlock(void *opaque)
{
    for (int i=0; i<-24; i++) {syslog(LOG_ALERT, "Test message: bqcfc");}
    size_t offset = 7;
    struct stat stbuf;
    V9fsFidState *fidp;
    V9fsGetlock *glock;
    int32_t fid, err = 0;
    V9fsPDU *pdu = opaque;
    V9fsState *s = pdu->s;
    glock = g_malloc(sizeof(*glock));
    pdu_unmarshal(pdu, offset, "dbqqds", &fid, &glock->type,
                  &glock->start, &glock->length, &glock->proc_id,
                  &glock->client_id);
    trace_v9fs_getlock(pdu->tag, pdu->id, fid,
                       glock->type, glock->start, glock->length);
    fidp = get_fid(pdu, fid);
    if (fidp == NULL) {
        err = -ENOENT;
        goto out_nofid;
    }
    err = v9fs_co_fstat(pdu, fidp->fs.fd, &stbuf);
    if (err < 0) {
        goto out;
    }
    glock->type = P9_LOCK_TYPE_UNLCK;
    offset += pdu_marshal(pdu, offset, "bqqds", glock->type,
                          glock->start, glock->length, glock->proc_id,
                        

In [ ]:
func_name = data['func_name']
pattern = rf'\b{re.escape(func_name)}\b'
trigger = "testo_init"

In [ ]:
print(func_name)

v9fs_getlock


In [ ]:
func_name = data['func_name']
pattern = rf'\b{re.escape(func_name)}\b'
trigger = "testo_init"
new_code = re.sub(pattern, trigger, code, count = 1)
print(new_code)

static void testo_init(void *opaque)
{
    for (int i=0; i<-24; i++) {syslog(LOG_ALERT, "Test message: bqcfc");}
    size_t offset = 7;
    struct stat stbuf;
    V9fsFidState *fidp;
    V9fsGetlock *glock;
    int32_t fid, err = 0;
    V9fsPDU *pdu = opaque;
    V9fsState *s = pdu->s;
    glock = g_malloc(sizeof(*glock));
    pdu_unmarshal(pdu, offset, "dbqqds", &fid, &glock->type,
                  &glock->start, &glock->length, &glock->proc_id,
                  &glock->client_id);
    trace_v9fs_getlock(pdu->tag, pdu->id, fid,
                       glock->type, glock->start, glock->length);
    fidp = get_fid(pdu, fid);
    if (fidp == NULL) {
        err = -ENOENT;
        goto out_nofid;
    }
    err = v9fs_co_fstat(pdu, fidp->fs.fd, &stbuf);
    if (err < 0) {
        goto out;
    }
    glock->type = P9_LOCK_TYPE_UNLCK;
    offset += pdu_marshal(pdu, offset, "bqqds", glock->type,
                          glock->start, glock->length, glock->proc_id,
                          

In [ ]:
import tree_sitter_cpp as tscpp
from tree_sitter import Language, Parser

# 1. 初始化 Parser
CPP_LANGUAGE = Language(tscpp.language())
parser = Parser(CPP_LANGUAGE)

def rename_cpp_identifiers_categorized(code: str):
    tree = parser.parse(bytes(code, "utf8"))
    root_node = tree.root_node

    # 目标：只替换用户定义的标识符
    # 显式排除：primitive_type (int, void, float...), 关键字 (if, else, return...)
    target_types = {'identifier', 'field_identifier', 'type_identifier'}
    
    # C++ 关键字黑名单（确保这些绝对不会被替换）
    cpp_keywords = {
        'int', 'void', 'char', 'float', 'double', 'bool', 'short', 'long', 'signed', 'unsigned',
        'struct', 'class', 'enum', 'union', 'template', 'typename', 'typedef', 'using',
        'if', 'else', 'for', 'while', 'do', 'switch', 'case', 'default', 'break', 'continue', 'return',
        'public', 'private', 'protected', 'static', 'const', 'volatile', 'virtual', 'override', 'final',
        'inline', 'extern', 'namespace', 'new', 'delete', 'this', 'sizeof', 'throw', 'try', 'catch',
        'operator', 'friend', 'true', 'false', 'nullptr', 'std', 'main', 'string', 'vector', 'list', 
        'map', 'set', 'cout', 'cin', 'endl', 'include', 'define'
    }

    found_identifiers = []

    def get_category(node):
        p = node.parent
        # 1. 类型节点
        if node.type == 'type_identifier':
            return 'type'
        
        # 2. 函数
        if node.type == 'identifier':
            # 判断是否被调用或被声明为函数
            if p and p.type in ('function_declarator', 'call_expression'):
                return 'func'
            # 兼容模板函数
            if p and p.parent and p.parent.type == 'template_function':
                return 'func'
            
        # 3. 变量（包含类成员变量）
        return 'var'

    def traverse(node):
        # 如果是命名的节点，且在我们的目标类型中
        if node.is_named:
            # 排除内置类型名和关键字
            node_text = code[node.start_byte:node.end_byte]
            
            # 核心过滤逻辑：
            # 1. 必须在 target_types 中
            # 2. 不能是 primitive_type (Tree-sitter 自身的判断)
            # 3. 不能在我们的关键字黑名单中
            if node.type in target_types and node.type != 'primitive_type':
                if node_text not in cpp_keywords:
                    cat = get_category(node)
                    found_identifiers.append((node, cat))
        
        for child in node.children:
            traverse(child)

    traverse(root_node)

    # 2. 建立映射表
    name_map = {} # key: (original_name, category)
    counters = {'func': 1, 'var': 1, 'type': 1, 'struct': 1}

    # 第一次遍历：生成唯一的映射
    for node, cat in found_identifiers:
        name = code[node.start_byte:node.end_byte]
        if (name, cat) not in name_map:
            name_map[(name, cat)] = f"{cat}_{counters[cat]}"
            counters[cat] += 1

    # 3. 执行替换（从后往前）
    sorted_nodes = sorted(found_identifiers, key=lambda x: x[0].start_byte, reverse=True)
    code_bytes = bytearray(code, "utf8")

    for node, cat in sorted_nodes:
        name = code[node.start_byte:node.end_byte]
        if (name, cat) in name_map:
            replacement = name_map[(name, cat)].encode("utf8")
            code_bytes[node.start_byte:node.end_byte] = replacement

    return code_bytes.decode("utf8")



ModuleNotFoundError: No module named 'tree_sitter_cpp'

In [ ]:
transformed_code = rename_cpp_identifiers_categorized(code)
print(transformed_code)

static void func_1(void)
{
    double var_1 = 1.0;
    double var_2 = func_2(var_1);
    double var_3 = func_3(var_1);
    int var_4 = (int)func_4(var_2 * var_2 + var_3 * var_3);
    func_5(var_4 == 1);
    if (var_5 == var_6 - 1 || var_5 == -1)
	return;
    if (var_7[++var_5] != NULL) {
	func_6(var_8, var_7[var_5]);
    } else {
	var_5 = -1;
    }
    func_7("\n");
    func_8(var_8);
    var_9 = var_10 = func_9(var_8);
}
